# Code was run on Colab Pro

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import collections
import torch.optim as optim
from torch.optim import Optimizer
import time
import matplotlib.pyplot as plt

from AdamW          import AdamW
from utils          import utility, misreportUtility, misreportOptimization, trueUtility, loss
from networks       import AdditiveMechanism, Misreports,AllocationNet,PaymentNet
from restrictedAdam import Adam 
from networks import MixedWrapper

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.cuda.set_device(2)

# Set Random Seed 

In [ ]:
# Initializing seeds
torch.manual_seed(0)
np.random.seed(0)

# Testing Function

In [ ]:
def test(nBatch, nbrInit, R, gamma=0.001, minimum=0, maximum=1):
    """
    直接输出两种结果：纯神经网络(Net) 与 混合机制(Mixed)
    - 不改你的 misreportUtility / utility / loss 实现
    - Mixed 评估时自动关闭直通（硬选择）
    """

    # ----- 测试数据 -----
    reserve=0.5
    true = np.random.rand(nBatch, nAgent, nObject)
    localMisreports     = np.random.rand(nBatch, nbrInit, nAgent, nObject)
    batchMisreports     = torch.tensor(localMisreports).float().to(device)
    batchTrueValuations = torch.tensor(true).float().to(device)
    batchMisreports.requires_grad = True

    def _eval_once(mech_callable):
        # 复制你 test() 的流程：误报优化 -> regret/payment/loss
        opt = Adam([batchMisreports], lr=gamma)
        for k in range(R):
            advU = misreportUtility(mech_callable, batchTrueValuations, batchMisreports)
            los  = -1*torch.mean(advU).to(device)
            los.backward()
            opt.step(restricted=True, min=minimum, max=maximum)
            opt.zero_grad()

        misReportUtilityMax  = torch.max(advU, dim=1)[0]
        allocation, payment = mech_callable(batchTrueValuations)
        regret = F.relu(misReportUtilityMax - utility(batchTrueValuations, allocation, payment))
        mregret = torch.sum(torch.mean(regret, dim=0)).to(device)
        mregret = float(mregret.detach().cpu().numpy())
        with torch.no_grad():
            l, rMean, p = loss(payment, regret)
        return mregret, float(p.detach().cpu().numpy()), float((-l).detach().cpu().numpy())**2

    # ----- 定义两个“机制可调用” -----
    # 纯 Net
    if hasattr(mechanism, "base"):   # 你用过 MixedWrapper 的情况
        def mech_net(X):  # 纯神经网络
            return mechanism.base(X)
    else:
        def mech_net(X):  # 没包过就直接用自身
            return mechanism(X)

    # Mixed（逐样本收益择优，评估时关直通）
    st_backup = getattr(mechanism, "st", None)
    if hasattr(mechanism, "st"):
        mechanism.st = False
    def mech_mix(X):
        return mechanism(X)
    # --------------------

    # ----- 分别评估两套 -----
    net_reg, net_pay, net_optrev   = _eval_once(mech_net)
    mix_reg, mix_pay, mix_optrev   = _eval_once(mech_mix)
    testRegret.append(net_reg/nAgent)
    testPayment.append(net_pay)
    testOptimal.append(net_optrev)

    # 恢复直通状态（若有）
    if hasattr(mechanism, "st"):
        mechanism.st = st_backup if st_backup is not None else True

    # 打印两行，简单明了
    print(f"[NET  ] regret={net_reg:.5f}  avg/bidder={net_reg/nAgent:.5f}  optRev={net_optrev:.3f}  payment={net_pay:.3f}")
    print(f"[MIXED] regret={mix_reg:.5f}  avg/bidder={mix_reg/nAgent:.5f}  optRev={mix_optrev:.3f}  payment={mix_pay:.3f}")

    # 返回字典（你要存表就用它）
    return {
        "net":   {"regret": net_reg, "payment": net_pay, "optrev": net_optrev},
        "mixed": {"regret": mix_reg, "payment": mix_pay, "optrev": mix_optrev},
    }
    testRegret.append(net_reg)
    testPayment.append(net_pay)
    testOptimal.append(net_optrev)


# Initializing Networks

In [ ]:
nAgent   = 2
nObject  = 2

# Parameters for the mechanism (payment and allocation network)
nLayersAllocation   = 5
nLayersPayment      = 5
widthAllocation     = 100
widthPayment        = 100

# Parameters for the misreport network
nLayersMisreport    = 5
widthMisreport      = 100

gamma              = 0.001 
testBatch          = 10000

nExperiments       = 200000
batchSize          = 500
nbrBatches         = int(nExperiments/batchSize)


mechanism_base            = AdditiveMechanism(nAgent, nObject, nLayersAllocation, widthAllocation).to(device)
mechanism_base= torch.load("2_2.pt")
mechanism = MixedWrapper(mechanism_base, reserve=0.5, straight_through=True).to(device)
mechanism.train() 
optimizerMechanism   = AdamW(mechanism_base .parameters(), lr=0.0001)

misreport            = Misreports(nAgent,nObject,nLayersMisreport, widthMisreport).to(device)
optimizerMisreport   = AdamW(misreport.parameters(), lr=0.0005)

In [ ]:
testRegret    = []
testMaxRegret = []
testPayment   = []
testOptimal   = []
testTime      = []
testIteration = [0]

# range of valuations
minimum            = 0
maximum            = 1

# Training

In [ ]:
duration   = 0
R          = 100

i=0

print("Initial Test")
test(50, nbrInit=300, R=300, gamma=0.001, minimum=0, maximum=1)

for t in range(1,60*nbrBatches+1):
    
    # Reinitialize Misreport network periodically at the beginning of training
    if (t%(2*nbrBatches) ==1):
      if   t< 20*nbrBatches+2 :
    
        misreport            = Misreports(nAgent,nObject,nLayersMisreport, widthMisreport).to(device)
        optimizerMisreport   = AdamW(misreport.parameters(), lr=0.001)

    batchTrueValuations = torch.tensor(np.random.rand(batchSize,nAgent,nObject)).float().to(device)
    
    # Optimize Misreport Network for R steps
    for k in range(R):
  
        misreports          = misreport(batchTrueValuations).unsqueeze(1)
        mUtility            = misreportUtility(mechanism,batchTrueValuations,misreports).squeeze(1)
        mLoss               = torch.sum(torch.mean(-mUtility,dim=0))

        optimizerMisreport.zero_grad()
        mLoss.backward()
        optimizerMisreport.step()

    
    # Optimize Mechanism network for one step
    misreports          = misreport(batchTrueValuations).unsqueeze(1)
    mUtility            = misreportUtility(mechanism,batchTrueValuations,misreports).squeeze(1)

    allocation, payment = mechanism(batchTrueValuations)

    regret     = 5*F.relu(mUtility -utility(batchTrueValuations, allocation, payment))
    l,rMean,p = loss(payment, regret)
        
    optimizerMechanism.zero_grad()

    l.backward()

    optimizerMechanism.step()
    
    # Test mechanism periodically
    if t % (2*nbrBatches)==0 :
        print("Batch: ", 2*int(t/(2*nbrBatches)))
        testTime.append(duration)
        testIteration.append(t/nbrBatches)
        test(50, nbrInit=300, R=300, gamma=0.001, minimum=0, maximum=1)

# Testing

In [ ]:
for i in range(50):
    test(200, nbrInit=1000, R=2000, gamma=0.001, minimum=0, maximum=1)

In [ ]:
totalregret = np.mean(np.array(testRegret[-200:]))
revenue     = np.mean(np.array(testPayment[-200:]))
print("Final Result")
print("Total Regret = ", '{0:.5f}'.format(totalregret), "Average regret per bidder: ",'{0:.5f}'.format(totalregret/nAgent), " Optimal Revenue: ",'{0:.3f}'.format(float(np.sqrt(revenue)-np.sqrt(totalregret))**2), " payment: ",'{0:.3f}'.format(revenue))

In [ ]:
stdregret = np.std(np.array(testRegret[-200:]))
stdrevenue= np.std(np.array(testPayment[-200:]))
print("std Regret = ", '{0:.5f}'.format(stdregret), "std regret per bidder: ",'{0:.5f}'.format(stdregret/nAgent), " std payment: ",'{0:.3f}'.format(stdrevenue))